**Exploritory Data Analysis**

In [2]:
import pandas as pd


""" --- Exploratory Data Analysis --- """
df = pd.read_csv("Data/IMDB.csv")

# Sample Head of dataset
print(df.head(5))
print(df.tail(5))

# Dataset shape and Description
print("\n",df.shape)
print("\n --- Dataset Description --- ")
print(df.describe())
print("-----------------------------")

# Dataset information
print("\n --- Dataset Summary --- ")
print(df.info())
print("-----------------------------")

# Dataset Null Values
print("\n --- Missing Values --- ")
print(df.isnull().sum())
print("-----------------------------")

# Dataset Duplicates
print("\n --- Duplicate Values --- ")
print(df[df.duplicated(subset=['review'])])
print("-----------------------------")

""" --- Initial Cleaning before EDA ---"""
# Droping Duplicates
df = df.drop_duplicates(subset=['review'])
print(df.shape)



                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
                                                  review sentiment
49995  I thought this movie did a down right good job...  positive
49996  Bad plot, bad dialogue, bad acting, idiotic di...  negative
49997  I am a Catholic taught in parochial elementary...  negative
49998  I'm going to have to disagree with the previou...  negative
49999  No one expects the Star Trek movies to be high...  negative

 (50000, 2)

 --- Dataset Description --- 
                                                   review sentiment
count                                               50000     50000
unique                                  

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import os
#import nltk
#nltk.download("stopwords", quiet=True)
#from nltk.corpus import stopwords

''' --- CONFIGURATIONS --- '''
OUTPUT_DIR = "eda_outputs"
TOP_N_WORDS = 25
TOP_N_BIGRAMS = 20

sns.set_style("whitegrid")
os.makedirs(OUTPUT_DIR, exist_ok=True)


POS_COLOR, NEG_COLOR = "#2ecc71", "#e74c3c"

HTML_TAG_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"http\S+|www\.\S+")
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s]")
ALLCAPS_WORD_RE = re.compile(r"\b[A-Z]{2,}\b")

def save_fig(fig, name):
    path = os.path.join(OUTPUT_DIR, name)
    fig.savefig(path, bbox_inches="tight")
    print(f"  -> saved {path}")
    plt.close(fig)


def clean_text(text):
    """Lowercase, strip HTML/URLs/punctuation/digits ."""
    text = HTML_TAG_RE.sub(" ", text)
    text = URL_RE.sub(" ", text)
    text = text.lower()
    text = NON_ALPHA_RE.sub(" ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize(text, min_len=3, remove_stopwords=True):
    tokens = clean_text(text).split()
    if remove_stopwords:
        tokens = [t for t in tokens]
    return tokens

''' --- CLASS IMBALANCE --- '''

print("\n"+"=" * 70)
print(" --- CLASS IMBALANCE --- ")
print("=" * 70)
counts = df["sentiment"].value_counts()
pct = (counts / counts.sum() * 100).round(2)
print(counts)
print(f"Percentages:\n{pct}")

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.barplot(
    x=counts.index, y=counts.values,
    hue=counts.index, legend=False,
    palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=ax,
)
for i, v in enumerate(counts.values):
    ax.text(i, v + max(counts.values) * 0.01, f"{v:,} ({pct.iloc[i]}%)",
            ha="center", fontweight="bold")
ax.set_title("Class Distribution: Positive vs Negative Reviews", fontsize=13, fontweight="bold")
ax.set_xlabel("Sentiment")
ax.set_ylabel("Number of Reviews")
save_fig(fig, "class_balance.png")





 --- CLASS IMBALANCE --- 
sentiment
positive    24884
negative    24698
Name: count, dtype: int64
Percentages:
sentiment
positive    50.19
negative    49.81
Name: count, dtype: float64
  -> saved eda_outputs\class_balance.png


In [4]:

''' --- WORD COUNT PER REVIEW --- '''

print("\n" + "=" * 70)
print(" WORD COUNT DISTRIBUTION")
print("=" * 70)
df["word_count"] = df["review"].str.split().apply(len)
print(df["word_count"].describe())

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.histplot(df, x="word_count", hue="sentiment", bins=50, kde=True,
             palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=axes[0])
axes[0].set_title("Distribution of Review Word Counts")
axes[0].set_xlabel("Word Count")
axes[0].set_xlim(0, df["word_count"].quantile(0.99))

sns.boxplot(data=df, x="sentiment", y="word_count",
            hue="sentiment", legend=False,
            palette={"positive": POS_COLOR, "negative": NEG_COLOR}, ax=axes[1])
axes[1].set_title("Word Count by Sentiment (Boxplot)")
axes[1].set_ylim(0, df["word_count"].quantile(0.99))
save_fig(fig, "word_count_dist.png")



 WORD COUNT DISTRIBUTION
count    49582.000000
mean       231.350167
std        171.542020
min          4.000000
25%        126.000000
50%        173.000000
75%        281.000000
max       2470.000000
Name: word_count, dtype: float64
  -> saved eda_outputs\word_count_dist.png


In [5]:

''' --- NOISE IDENTIFICATION --- '''

print("\n" + "=" * 70)
print(" NOISE IDENTIFICATION")
print("=" * 70)
has_url = df["review"].str.contains(URL_RE, regex=True)
has_html = df["review"].str.contains(HTML_TAG_RE, regex=True)
has_digit = df["review"].str.contains(r"\d", regex=True)
has_special = df["review"].str.contains(r"[^a-zA-Z0-9\s\.\,\!\?\'\"]", regex=True)
has_allcaps = df["review"].str.contains(ALLCAPS_WORD_RE, regex=True)

noise_summary = pd.Series({
    "Contains URL": has_url.sum(),
    "Contains HTML tags": has_html.sum(),
    "Contains digits": has_digit.sum(),
    "Contains special chars": has_special.sum(),
    "Contains ALL-CAPS word": has_allcaps.sum(),
})
print(noise_summary)
print(f"\nExample with HTML tags:\n{df.loc[has_html, 'review'].iloc[0][:200]}...")
if has_url.any():
    print(f"\nExample with URL:\n{df.loc[has_url, 'review'].iloc[0][:200]}...")

fig, ax = plt.subplots(figsize=(7, 4.5))
sns.barplot(x=noise_summary.values, y=noise_summary.index, hue=noise_summary.index,
            legend=False, palette="rocket", ax=ax)
for i, v in enumerate(noise_summary.values):
    ax.text(v + 300, i, f"{v:,} ({v/len(df)*100:.1f}%)", va="center")
ax.set_title("Noise Patterns Found in Raw Reviews", fontsize=13, fontweight="bold")
ax.set_xlabel("Number of Reviews Affected")
save_fig(fig, "noise_summary.png")


 NOISE IDENTIFICATION
Contains URL                196
Contains HTML tags        28968
Contains digits           27786
Contains special chars    44635
Contains ALL-CAPS word    23982
dtype: int64

Example with HTML tags:
One of the other reviewers has mentioned that after watching just 1 Oz episode you'll be hooked. They are right, as this is exactly what happened with me.<br /><br />The first thing that struck me abo...

Example with URL:
Mario Lewis of the Competitive Enterprise Institute has written a definitive 120-page point-by-point, line-by-line refutation of this mendacious film, which should be titled A CONVENIENT LIE. The webs...
  -> saved eda_outputs\noise_summary.png


In [6]:
from collections import Counter

''' --- MOST FREQUENT WORDS --- '''

print("\n" + "=" * 70)
print(" MOST FREQUENT WORDS")
print("=" * 70)
all_tokens = [tok for review in df["review"] for tok in tokenize(review)]
word_freq = Counter(all_tokens)
top_words = word_freq.most_common(TOP_N_WORDS)
print(f"Total tokens : {len(all_tokens):,} | Unique words: {len(word_freq):,}")
print(f"Top {TOP_N_WORDS}: {top_words}")

words, freqs = zip(*top_words)
fig, ax = plt.subplots(figsize=(8, 9))
sns.barplot(x=list(freqs), y=list(words), hue=list(words), legend=False,
            palette="viridis", ax=ax)
ax.set_title(f"Top {TOP_N_WORDS} Most Frequent Words (Whole Dataset)", fontsize=13, fontweight="bold")
ax.set_xlabel("Frequency")
save_fig(fig, "top_words.png")


 MOST FREQUENT WORDS
Total tokens : 11,620,159 | Unique words: 99,293
Top 25: [('the', 662922), ('and', 322089), ('a', 320804), ('of', 287341), ('to', 266119), ('is', 209640), ('it', 189442), ('in', 185420), ('i', 174133), ('this', 149809), ('that', 142816), ('s', 128908), ('was', 94814), ('as', 91090), ('movie', 87276), ('for', 86821), ('with', 86717), ('but', 82922), ('film', 79148), ('you', 68523), ('t', 67750), ('on', 67530), ('not', 60273), ('he', 58527), ('are', 57964)]
  -> saved eda_outputs\top_words.png


In [7]:

''' --- RARE WORD ANALYSIS --- '''

print("\n" + "=" * 70)
print(" RARE WORD ANALYSIS")
print("=" * 70)
freq_counts = Counter(word_freq.values())
once = freq_counts[1]
twice = freq_counts[2]
vocab_size = len(word_freq)
print(f"Vocabulary size: {vocab_size:,}")
print(f"Words occurring exactly once : {once:,} ({once/vocab_size*100:.1f}% of vocab)")
print(f"Words occurring exactly twice: {twice:,} ({twice/vocab_size*100:.1f}% of vocab)")
print(f"Words occurring <= 2 times combined: {once+twice:,} ({(once+twice)/vocab_size*100:.1f}% of vocab)")

freq_values = np.array(list(word_freq.values()))
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

bins_labels = ["1", "2", "3-5", "6-10", "11-50", "50+"]
bins_counts = [
    (freq_values == 1).sum(),
    (freq_values == 2).sum(),
    ((freq_values >= 3) & (freq_values <= 5)).sum(),
    ((freq_values >= 6) & (freq_values <= 10)).sum(),
    ((freq_values >= 11) & (freq_values <= 50)).sum(),
    (freq_values > 50).sum(),
]
sns.barplot(x=bins_labels, y=bins_counts, hue=bins_labels, legend=False,
            palette="mako", ax=axes[0])
for i, v in enumerate(bins_counts):
    axes[0].text(i, v + max(bins_counts) * 0.01, f"{v:,}", ha="center", fontsize=9)
axes[0].set_title("Vocabulary Split by Word Frequency")
axes[0].set_xlabel("Frequency Bucket (occurrences in corpus)")
axes[0].set_ylabel("Number of Unique Words")

sns.histplot(freq_values, bins=60, ax=axes[1], color="#34495e")
axes[1].set_yscale("log")
axes[1].set_title("Word-Frequency Distribution (log scale)")
axes[1].set_xlabel("Occurrences per Word")
axes[1].set_ylabel("Number of Words (log)")
save_fig(fig, "word_freq_distribution.png")


 RARE WORD ANALYSIS
Vocabulary size: 99,293
Words occurring exactly once : 37,524 (37.8% of vocab)
Words occurring exactly twice: 11,677 (11.8% of vocab)
Words occurring <= 2 times combined: 49,201 (49.6% of vocab)
  -> saved eda_outputs\word_freq_distribution.png


In [8]:
from sklearn.feature_extraction.text import CountVectorizer
from wordcloud import WordCloud

''' --- WORD CLOUDS + TOP BIGRAMS PER CLASS --- '''

print("\n" + "=" * 70)
print("WORD CLOUDS & TOP BIGRAMS PER CLASS")
print("=" * 70)
def joined_tokens(reviews):
    return " ".join(" ".join(tokenize(r)) for r in reviews)

pos_text = joined_tokens(df.loc[df["sentiment"] == "positive", "review"])
neg_text = joined_tokens(df.loc[df["sentiment"] == "negative", "review"])

# Word clouds
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, text, title, cmap in [
    (axes[0], pos_text, "Positive Reviews", "Greens"),
    (axes[1], neg_text, "Negative Reviews", "Reds"),
]:
    wc = WordCloud(width=800, height=500, background_color="white",
                    colormap=cmap, max_words=120).generate(text)
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    ax.set_title(title, fontsize=14, fontweight="bold")
save_fig(fig, "wordclouds.png")

# Top bigrams per class
def top_bigrams(texts, n=TOP_N_BIGRAMS):
    cleaned = [clean_text(t) for t in texts]
    vec = CountVectorizer(ngram_range=(2, 2), stop_words="english", min_df=3)
    X = vec.fit_transform(cleaned)
    sums = np.asarray(X.sum(axis=0)).ravel()
    pairs = sorted(zip(vec.get_feature_names_out(), sums), key=lambda x: -x[1])[:n]
    return pairs

pos_bigrams = top_bigrams(df.loc[df["sentiment"] == "positive", "review"])
neg_bigrams = top_bigrams(df.loc[df["sentiment"] == "negative", "review"])
print(f"Top positive bigrams: {pos_bigrams[:10]}")
print(f"Top negative bigrams: {neg_bigrams[:10]}")

fig, axes = plt.subplots(1, 2, figsize=(13, 7))
for ax, pairs, title, color in [
    (axes[0], pos_bigrams, "Top Bigrams — Positive Reviews", POS_COLOR),
    (axes[1], neg_bigrams, "Top Bigrams — Negative Reviews", NEG_COLOR),
]:
    bg, cnt = zip(*pairs)
    sns.barplot(x=list(cnt), y=list(bg), color=color, ax=ax)
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Frequency")
save_fig(fig, "top_bigrams.png")



WORD CLOUDS & TOP BIGRAMS PER CLASS
  -> saved eda_outputs\wordclouds.png
Top positive bigrams: [('ve seen', np.int64(1851)), ('don know', np.int64(897)), ('new york', np.int64(854)), ('special effects', np.int64(804)), ('good movie', np.int64(759)), ('real life', np.int64(759)), ('years ago', np.int64(740)), ('great movie', np.int64(720)), ('year old', np.int64(685)), ('sci fi', np.int64(623))]
Top negative bigrams: [('ve seen', np.int64(2282)), ('special effects', np.int64(1434)), ('waste time', np.int64(1434)), ('don know', np.int64(1291)), ('looks like', np.int64(1248)), ('low budget', np.int64(1223)), ('movie just', np.int64(1067)), ('look like', np.int64(950)), ('bad movie', np.int64(918)), ('year old', np.int64(912))]
  -> saved eda_outputs\top_bigrams.png


In [9]:
import re
from bs4 import BeautifulSoup
import contractions

''' --- Binary Label Encoding --- '''

label_map = {"positive": 1, "negative": 0}
df["sentiment"] = df["sentiment"].map(label_map)

""" --- Text Cleaning Pipeline --- """

def clean_text(text: str) -> str:

    # Strip HTML tags
    text = BeautifulSoup(text, "html.parser").get_text()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Expand contractions
    text = contractions.fix(text)

    # Lowercase
    text = text.lower()

    # Scrub digits
    text = re.sub(r"\d+", " ", text)

    # Strip special characters and symbols
    text = re.sub(r"[^a-z\s]", " ", text)

    # Collapse whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text

df["review"] = df["review"].apply(clean_text)

# Sanity Check

print("Label distribution ")
print(df["sentiment"].value_counts())
print("\n Sample cleaned review ")
print("-------------------------")
print(df["review"].iloc[0])

Label distribution 
sentiment
1    24884
0    24698
Name: count, dtype: int64

 Sample cleaned review 
-------------------------
one of the other reviewers has mentioned that after watching just oz episode you will be hooked they are right as this is exactly what happened with me the first thing that struck me about oz was its brutality and unflinching scenes of violence which set in right from the word go trust me this is not a show for the faint hearted or timid this show pulls no punches with regards to drugs sex or violence its is hardcore in the classic use of the word it is called oz as that is the nickname given to the oswald maximum security state penitentary it focuses mainly on emerald city an experimental section of the prison where all the cells have glass fronts and face inwards so privacy is not high on the agenda them city is home to many aryans muslims gangstas latinos christians italians irish and more so scuffles death stares dodgy dealings and shady agreements are ne

In [10]:
from sklearn.model_selection import train_test_split

""" --- Train / Validation / Test Split (70 / 15 / 15) --- """

X = df["review"]
y = df["sentiment"]

# Step 1: Carve out 70% train, 30% temp (val + test combined)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y          # preserves 50/50 class balance in every split
)

# Split the 30% temp evenly → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

# Split Result
total = len(df)

print("=" * 55)
print(f"{'Split':<15} {'Samples':>10} {'Share':>8}  {'Pos ratio':>10}")
print("=" * 55)
for name, X_s, y_s in [
    ("Train",      X_train, y_train),
    ("Validation", X_val,   y_val),
    ("Test",       X_test,  y_test),
]:
    print(f"{name:<15} {len(X_s):>10,} {len(X_s)/total*100:>7.1f}%  {y_s.mean():>10.4f}")
print("=" * 55)
print(f"{'Total':<15} {total:>10,} {'100.0%':>8}")

Split              Samples    Share   Pos ratio
Train               34,707    70.0%      0.5019
Validation           7,437    15.0%      0.5018
Test                 7,438    15.0%      0.5019
Total               49,582   100.0%
